In [2]:
pip install pandas psycopg2-binary python-dotenv 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd

In [ ]:
import json
with open(r"C:\Users\HP\Desktop\Building with oyin\Multi_Format_Warehouse\product_reviews.json") as f:
    data = json.load(f)



In [5]:
print(type(data))
print(data.keys())
print(type(data["products"]))
print(len(data["products"]))

<class 'dict'>
dict_keys(['generated_at', 'source', 'products'])
<class 'list'>
10


In [6]:
print(type(data["products"][0]))
print(data["products"][0].keys())
print(data["products"][2].keys())
print(data["products"][0]["reviews"])
print(data["products"][2]["reviews"])

<class 'dict'>
dict_keys(['product_id', 'product_name', 'category', 'reviews'])
dict_keys(['product_id', 'product_name', 'category', 'reviews'])
[]
[{'review_id': 'REV0001', 'customer_id': 'CUST209', 'rating': 1, 'comment': 'Good value for money', 'review_date': '2025-12-17'}, {'review_id': 'REV0002', 'customer_id': 'CUST229', 'rating': 1, 'comment': 'Arrived late but fine', 'review_date': 'January 15, 2026'}]


Make two empty lists, product_rows and review_rows.
Loop over data["products"]. For each product, append a dict with just its three product fields to product_rows.
Inside that loop, add a second loop over the product’s reviews. For each review, add its product_id to the review dict, then append it to review_rows.
Turn both lists into DataFrames with pd.DataFrame(...).
Print .shape and .head() of each.

In [7]:
product_rows = []
review_rows = []

for product in data["products"]:
    product_rows.append({
        "product_id": product["product_id"],
        "product_name": product["product_name"],
        "category": product["category"],
    })
    for review in product["reviews"]:
        review["product_id"] = product["product_id"]
        review_rows.append(review)

products = pd.DataFrame(product_rows)
reviews = pd.DataFrame(review_rows)

print(products.shape, reviews.shape)
reviews.head()

(10, 3) (31, 6)


,review_id,customer_id,rating,comment,review_date,product_id
0,REV0001,CUST209,1,Good value for money,2025-12-17,P003
1,REV0002,CUST229,1,Arrived late but fine,"January 15, 2026",P003
2,REV0003,CUST222,6,Very sturdy,"February 17, 2025",P004
3,REV0004,CUST227,4,Stopped working after a week,2025-07-06,P004
4,REV0005,CUST225,5,Arrived late but fine,2025-01-27,P004


In [8]:
print(reviews["review_id"].duplicated().sum())
print(reviews["rating"].value_counts().sort_index())

3
rating
1    4
2    5
3    3
4    7
5    9
6    3
Name: count, dtype: int64


In [9]:
rows_in = len(reviews)


reviews_dedup = reviews.drop_duplicates(subset=["review_id"], keep="first")
dupes_removed = rows_in - len(reviews_dedup)

mask = reviews_dedup["rating"].between(1, 5)
reviews_valid = reviews_dedup[mask].copy()
reviews_rejected = reviews_dedup[~mask].copy()
reviews_rejected["reason"] = "invalid rating"

print(rows_in, dupes_removed, len(reviews_valid), len(reviews_rejected))
assert rows_in == dupes_removed + len(reviews_valid) + len(reviews_rejected), (
    f"Row mismatch: in={rows_in}, dupes={dupes_removed}, "
    f"valid={len(reviews_valid)}, rejected={len(reviews_rejected)}"
)

31 3 26 2


In [10]:
print(reviews["review_date"].dtype)

str


In [11]:
formats = ("%Y-%m-%d", "%d/%m/%Y", "%B %d, %Y")

def parse_date(s):
    for fmt in formats:
        result = pd.to_datetime(s, format=fmt, errors="coerce")
        if pd.notna(result):
            return result
    return pd.NaT

reviews_valid["review_date"] = reviews_valid["review_date"].apply(parse_date)



In [12]:
print(reviews_valid["review_date"].isna().sum())                     
print(reviews_valid[reviews_valid["review_id"].isin(["REV0002", "REV0007", "REV0028"])])
print(reviews_valid["review_date"].dtype)                              
print(reviews_valid[reviews_valid["review_id"] == "REV0009"])     

0
   review_id customer_id  rating                comment review_date product_id
1    REV0002     CUST229       1  Arrived late but fine  2026-01-15       P003
6    REV0007     CUST215       5            Very sturdy  2025-08-10       P004
30   REV0028     CUST220       2   Good value for money  2025-12-04       P010
datetime64[us]
  review_id customer_id  rating               comment review_date product_id
9   REV0009     CUST236       1  Exactly as described  2026-01-19       P005


In [13]:
products["category"].value_counts()

category
Accessories    3
accessories    1
audio          1
Furniture      1
home           1
ELECTRONICS    1
Stationery     1
Audio          1
Name: count, dtype: int64

In [14]:
products["category"] = products["category"].str.strip().str.lower()
products["category"].value_counts()

category
accessories    4
audio          2
furniture      1
home           1
electronics    1
stationery     1
Name: count, dtype: int64

In [ ]:
orders = pd.read_csv()
print(orders.shape)r"C:\Users\HP\Documents\multi format orders.csv"
print(orders.isna().sum())
print(orders['quantity'].duplicated().sum())
print(orders['product_id'].value_counts())

(310, 6)
order_id       0
customer_id    0
product_id     0
quantity       0
unit_price     8
order_date     0
dtype: int64
304
product_id
P005    38
P008    37
P009    32
P007    32
P003    30
P004    29
P006    28
P001    27
P010    26
P002    25
P050     3
P051     3
Name: count, dtype: int64


In [16]:
print(orders.duplicated().sum())
print(orders["quantity"].value_counts().sort_index())
print(orders["order_date"].head())

10
quantity
-1     3
 1    59
 2    56
 3    63
 4    63
 5    66
Name: count, dtype: int64
0      2/5/2025
1      5/8/2025
2     8/31/2025
3    10/10/2025
4     9/17/2025
Name: order_date, dtype: str


In [17]:
rows_in = len(orders)
orders_dedup = orders.drop_duplicates()
dupes_removed = rows_in - len(orders_dedup)
print(rows_in, dupes_removed, len(orders_dedup))


310 10 300


In [18]:
mask = orders_dedup["product_id"].isin(products["product_id"])
orders_valid_products = orders_dedup[mask].copy()
orders_orphans = orders_dedup[~mask].copy()
orders_orphans["reason"] = "product_id not found in products"

print(len(orders_valid_products), len(orders_orphans))
assert rows_in == dupes_removed + len(orders_valid_products) + len(orders_orphans)

294 6


In [19]:
mask = orders_valid_products["quantity"] >= 1
orders_valid_qty = orders_valid_products[mask].copy()
orders_bad_qty = orders_valid_products[~mask].copy()
orders_bad_qty["reason"] = "quantity must be at least 1"

print(len(orders_valid_qty), len(orders_bad_qty))
assert rows_in == (dupes_removed + len(orders_orphans)
                   + len(orders_bad_qty) + len(orders_valid_qty))

291 3


In [20]:
orders_clean = orders_valid_qty.copy()
orders_clean["price_imputed"] = orders_clean["unit_price"].isna()
means = orders_clean.groupby("product_id")["unit_price"].transform("mean")
orders_clean["unit_price"] = orders_clean["unit_price"].fillna(means)
orders_clean["unit_price"] = orders_clean["unit_price"].round(2)
print(orders_clean["unit_price"].isna().sum())   
print(orders_clean["price_imputed"].sum())     

0
8


In [21]:
print(orders_clean["order_date"])

0        2/5/2025
1        5/8/2025
2       8/31/2025
3      10/10/2025
4       9/17/2025
          ...    
305     6/14/2025
306     1/31/2026
307    10/16/2025
308     8/12/2025
309     2/27/2025
Name: order_date, Length: 291, dtype: str


In [22]:
orders_clean["order_date"] = pd.to_datetime(
    orders_clean["order_date"], format="%m/%d/%Y", errors="coerce"
)

In [23]:
print(orders_clean["order_date"].isna().sum())                 
print(orders_clean["order_date"].dtype)                        
print(orders_clean[orders_clean["order_date"].dt.day > 12].head())

0
datetime64[us]
   order_id customer_id product_id  quantity  unit_price order_date  \
2   ORD4045     CUST227       P003         2    43079.72 2025-08-31   
4   ORD4290     CUST207       P009         4    22333.18 2025-09-17   
5   ORD4190     CUST231       P007         1    13962.39 2025-12-25   
6   ORD4096     CUST233       P009         3    22781.43 2026-01-25   
11  ORD4284     CUST237       P003         2    44873.81 2025-05-27   

    price_imputed  
2           False  
4           False  
5            True  
6           False  
11          False  


In [24]:
orders_orphans["source"] = "orders"
orders_bad_qty["source"] = "orders"
reviews_rejected["source"] = "reviews"

errors = pd.concat(
    [orders_orphans, orders_bad_qty, reviews_rejected],
    ignore_index=True,
)

first = ["source", "reason"]
errors = errors[first + [c for c in errors.columns if c not in first]]

errors.to_csv("errors.csv", index=False)
assert len(errors) == 11, f"expected 11 rejected rows, got {len(errors)}"
errors

,source,reason,order_id,customer_id,product_id,quantity,unit_price,order_date,review_id,rating,comment,review_date
0,orders,product_id not found in products,ORD4032,CUST238,P050,2.0,12061.74,12/1/2025,NaN,NaN,NaN,NaN
1,orders,product_id not found in products,ORD4287,CUST223,P051,5.0,14571.87,7/21/2025,NaN,NaN,NaN,NaN
2,orders,product_id not found in products,ORD4135,CUST236,P051,5.0,9462.28,8/19/2025,NaN,NaN,NaN,NaN
3,orders,product_id not found in products,ORD4218,CUST236,P051,3.0,47026.17,7/24/2025,NaN,NaN,NaN,NaN
4,orders,product_id not found in products,ORD4085,CUST231,P050,5.0,13822.20,6/10/2025,NaN,NaN,NaN,NaN
5,orders,product_id not found in products,ORD4212,CUST238,P050,3.0,14368.05,3/15/2025,NaN,NaN,NaN,NaN
6,orders,quantity must be at least 1,ORD4099,CUST221,P010,-1.0,12595.31,11/16/2025,NaN,NaN,NaN,NaN
7,orders,quantity must be at least 1,ORD4260,CUST216,P005,-1.0,7484.88,5/27/2025,NaN,NaN,NaN,NaN
8,orders,quantity must be at least 1,ORD4185,CUST238,P007,-1.0,14013.73,3/13/2025,NaN,NaN,NaN,NaN
9,reviews,invalid rating,NaN,CUST222,P004,NaN,NaN,NaN,REV0003,6.0,Very sturdy,"February 17, 2025"


In [25]:
from extract import extract_products_and_reviews, extract_orders

products, reviews = extract_products_and_reviews(r"C:\Users\HP\Desktop\Building with oyin\Multi_Format_Warehouse\product_reviews.json")
orders = extract_orders(r"C:\Users\HP\Documents\multi format orders.csv")
print(products.shape, reviews.shape, orders.shape)

(10, 3) (31, 6) (310, 6)


In [26]:
from clean import clean_products, clean_reviews

products_c = clean_products(products)
reviews_c, reviews_rej, rev_dupes = clean_reviews(reviews, products_c["product_id"])
print(len(reviews_c), len(reviews_rej), rev_dupes)   
print(reviews_c["review_date"].dtype)                

26 2 3
datetime64[us]


In [27]:
from clean import clean_products, clean_reviews, clean_orders

orders_c, orders_rej, ord_dupes = clean_orders(orders, products_c["product_id"])
print(len(orders_c), len(orders_rej), ord_dupes)   
print(orders_c["unit_price"].isna().sum())          
print(orders_c["price_imputed"].sum())              

291 9 10
0
8
